In [ ]:
import os
import sys
from pathlib import Path
import sqlite3
import datetime as dt
import logging
from dataclasses import dataclass, field, asdict
import pandas as pd
import yaml
import uuid
import matplotlib
import matplotlib.colors
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
import seaborn as sns
import random
import numpy as np
import torch

In [ ]:
def logging_with_display(
    row, level=logging.INFO, display_func=print, except_logging=False
):
    if not except_logging:
        logging.log(level, row)
    display_func(row)


def set_seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
@dataclass
class TrainingConfig:
    # config file path itself
    config_path: str = field(
        default="training_config.yaml",
        metadata={"help": "Path to the config yaml file"},
    )

    # Hyperparameters and paths for learning
    learning_rate: float = field(
        default=0.001, metadata={"help": "Learning rate for the optimizer"}
    )
    batch_size: int = field(default=32, metadata={"help": "Batch size for training"})
    num_epochs: int = field(default=10, metadata={"help": "Number of training epochs"})
    log_save_path: str = field(
        default="output/logs", metadata={"help": "Path to save training logs"}
    )
    input_db_path: str = field(
        default="input/database.sqlite", metadata={"help": "Path to the input database"}
    )
    output_root: str = field(
        default="output/runs", metadata={"help": "Root directory for output runs"}
    )
    run_id: str = field(
        default_factory=lambda: f"{dt.datetime.now():%Y%m%d_%H%M%S}_{uuid.uuid4().hex[:6]}"
    )

    seed_id: int = field(
        default=42, metadata={"help": "Random seed for reproducibility"}
    )

    def __post_init__(self):
        if os.path.exists(self.config_path):
            self.load_config()
        run_dir = Path(self.output_root) / self.run_id
        run_dir.mkdir(parents=True, exist_ok=True)
        self.model_save_path = str(run_dir / "model.pth")
        self.log_save_path = str(run_dir)
        self.config_save_path = str(run_dir / "training_config.yaml")

        os.makedirs(self.log_save_path, exist_ok=True)
        logging.basicConfig(
            filename=os.path.join(self.log_save_path, "training.log"),
            format="%(asctime)s | %(levelname)s | %(message)s",
            level=logging.INFO,
        )

        self.save_runtime_config()

    def save_runtime_config(self):
        with open(self.config_save_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(asdict(self), f, allow_unicode=True, sort_keys=False)

    def load_config(self):
        with open(self.config_path, "r", encoding="utf-8") as f:
            config_data = yaml.safe_load(f) or {}

        for key, value in config_data.items():
            if hasattr(self, key):
                setattr(self, key, value)

In [ ]:
config = TrainingConfig()

set_seed_all(config.seed_id)

# open sqlite3 database
db_path = Path(config.input_db_path)
conn = sqlite3.connect(db_path)
logging_with_display(f"Connected to database at {db_path}")

# load data from database
query = "SELECT * FROM iris"
df_data = pd.read_sql_query(query, conn)
logging_with_display(f"Loaded {len(df_data)} records from the database")

# close the database connection
conn.close()
logging_with_display("Closed database connection")

In [ ]:
# EDA about the data
logging_with_display(f"Data shape: {df_data.shape}", except_logging=True)
logging_with_display(f"Data columns: {df_data.columns.tolist()}", except_logging=True)
logging_with_display(f"Data head:\n{df_data.head()}", except_logging=True)
logging_with_display(f"Data description:\n{df_data.describe()}", except_logging=True)

In [ ]:
# mapping species to numeric labels
species_unique = df_data["Species"].unique()
species_to_idx = {species: idx for idx, species in enumerate(species_unique)}
idx_to_species = {idx: species for species, idx in species_to_idx.items()}
df_data["species_label"] = df_data["Species"].map(species_to_idx)
logging_with_display(f"Species mapping: {species_to_idx}", except_logging=True)

# define features
numerical_features = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]
categorical_feature = "Species"

# split data into train and test sets
X = df_data.drop(columns=["species_label"])
y = df_data["species_label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=config.seed_id
)
logging_with_display(f"Train set size: {len(X_train)}")
logging_with_display(f"Test set size: {len(X_test)}")

In [ ]:
temp = dict(
    layout=go.Layout(font=dict(family="Malgun Gothic", size=12), height=500, width=1000)
)

### Distribution of species

In [ ]:
target = df_data["species_label"].value_counts()
target.rename(index=idx_to_species, inplace=True)
pal, color = ["#3A97E9", "#FA92FD", "#FCF48F"], ["#77B9F3", "#F9B1F0", "#FDF1A7"]
fig = go.Figure()
fig.add_trace(
    go.Pie(
        labels=target.index,
        values=target.values,
        hole=0.4,
        sort=False,
        showlegend=True,
        marker=dict(colors=color, line=dict(color=pal, width=2)),
        textinfo="value+percent",
        insidetextorientation="radial",
        hovertemplate="%{label}: %{value} (%{percent})",
    )
)
fig.update_layout(
    template=temp,
    title="Distribution of Target Classes",
    legend=dict(traceorder="reversed", y=1.05, x=0),
)
fig.show()

### Hist of features

In [ ]:
# 단변량 분석
# SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm
# 각 feature의 분포를 species_label별로 그리되, label은 species_label이 아닌 idx_to_species로 매핑된 문자열로 표시
fig, axes = plt.subplots(2, 2, figsize=(8, 6))
sns.kdeplot(data=df_data, x="SepalLengthCm", hue="Species", fill=True, ax=axes[0, 0])
sns.kdeplot(data=df_data, x="SepalWidthCm", hue="Species", fill=True, ax=axes[0, 1])
sns.kdeplot(data=df_data, x="PetalLengthCm", hue="Species", fill=True, ax=axes[1, 0])
sns.kdeplot(data=df_data, x="PetalWidthCm", hue="Species", fill=True, ax=axes[1, 1])
axes[0, 0].set_title("Sepal Length Distribution")
axes[0, 1].set_title("Sepal Width Distribution")
axes[1, 0].set_title("Petal Length Distribution")
axes[1, 1].set_title("Petal Width Distribution")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 5))
for i, feature in enumerate(numerical_features):
    sns.boxplot(
        x=categorical_feature,
        y=feature,
        data=df_data,
        ax=ax[i],
        hue=categorical_feature,
        palette=pal,
    )
    ax[i].set_title(f"{feature} by {categorical_feature}")
plt.tight_layout()
plt.show()

In [ ]:
# 이변량 분석
corr = df_data[numerical_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))[1:, :-1]
corr = corr.iloc[1:, :-1].copy()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", mask=mask, vmin=-1, vmax=1)
ax.tick_params(left=False, bottom=False)
ax.set_xticklabels(
    ax.get_xticklabels(), rotation=45, horizontalalignment="right", fontsize=12
)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=12)
plt.title("Correlation Heatmap of Numerical Features")
plt.show()

In [ ]:
# hexbin plot
fig, ax = plt.subplots(1, 4, figsize=(16, 5))
# corr between SepalLengthCm and SepalWidthCm
ax[0].hexbin(
    x=df_data["SepalLengthCm"],
    y=df_data["SepalWidthCm"],
    gridsize=20,
    cmap="Blues",
    mincnt=1,
)
ax[0].set(xlabel="Sepal Length (cm)", ylabel="Sepal Width (cm)")
ax[0].set_title("Sepal Length vs Sepal Width")
# corr between SepalLengthCm and PetalLengthCm
ax[1].hexbin(
    x=df_data["SepalLengthCm"],
    y=df_data["PetalLengthCm"],
    gridsize=20,
    cmap="Greens",
    mincnt=1,
)
ax[1].set(xlabel="Sepal Length (cm)", ylabel="Petal Length (cm)")
ax[1].set_title("Sepal Length vs Petal Length")
# corr between SepalLengthCm and PetalWidthCm
ax[2].hexbin(
    x=df_data["SepalLengthCm"],
    y=df_data["PetalWidthCm"],
    gridsize=20,
    cmap="Reds",
    mincnt=1,
)
ax[2].set(xlabel="Sepal Length (cm)", ylabel="Petal Width (cm)")
ax[2].set_title("Sepal Length vs Petal Width")
# corr between SepalWidthCm and PetalLengthCm
ax[3].hexbin(
    x=df_data["SepalWidthCm"],
    y=df_data["PetalLengthCm"],
    gridsize=20,
    cmap="Purples",
    mincnt=1,
)
ax[3].set(xlabel="Sepal Width (cm)", ylabel="Petal Length (cm)")
ax[3].set_title("Sepal Width vs Petal Length")
plt.tight_layout()
plt.show()

In [ ]:
# species 별로 hexbin plot
fig, ax = plt.subplots(3, 4, figsize=(16, 12))
for i, species in enumerate(species_unique):
    subset = df_data[df_data["Species"] == species]
    ax[i, 0].hexbin(
        x=subset["SepalLengthCm"],
        y=subset["SepalWidthCm"],
        gridsize=20,
        cmap="Blues",
        mincnt=1,
    )
    ax[i, 0].set(xlabel="Sepal Length (cm)", ylabel="Sepal Width (cm)")
    ax[i, 0].set_title(f"{species} - Sepal Length vs Sepal Width")
    ax[i, 1].hexbin(
        x=subset["SepalLengthCm"],
        y=subset["PetalLengthCm"],
        gridsize=20,
        cmap="Greens",
        mincnt=1,
    )
    ax[i, 1].set(xlabel="Sepal Length (cm)", ylabel="Petal Length (cm)")
    ax[i, 1].set_title(f"{species} - Sepal Length vs Petal Length")
    ax[i, 2].hexbin(
        x=subset["SepalLengthCm"],
        y=subset["PetalWidthCm"],
        gridsize=20,
        cmap="Reds",
        mincnt=1,
    )
    ax[i, 2].set(xlabel="Sepal Length (cm)", ylabel="Petal Width (cm)")
    ax[i, 2].set_title(f"{species} - Sepal Length vs Petal Width")
    ax[i, 3].hexbin(
        x=subset["SepalWidthCm"],
        y=subset["PetalLengthCm"],
        gridsize=20,
        cmap="Purples",
        mincnt=1,
    )
    ax[i, 3].set(xlabel="Sepal Width (cm)", ylabel="Petal Length (cm)")
    ax[i, 3].set_title(f"{species} - Sepal Width vs Petal Length")
plt.tight_layout()
plt.show()

In [ ]:
# lgbm 모델 학습을 위한 데이터 준비
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import ConfusionMatrixDisplay

X_train_lgbm = X_train[numerical_features]
X_test_lgbm = X_test[numerical_features]
y_train_lgbm = y_train
y_test_lgbm = y_test
lgbm_model = LGBMClassifier(
    n_estimators=100, learning_rate=0.1, random_state=config.seed_id
)
lgbm_model.fit(
    X_train_lgbm,
    y_train_lgbm,
    eval_set=[(X_test_lgbm, y_test_lgbm)],
    eval_metric="multi_logloss",
    callbacks=[early_stopping(stopping_rounds=10), log_evaluation(period=10)],
)
y_pred_lgbm = lgbm_model.predict(X_test_lgbm)
accuracy = accuracy_score(y_test_lgbm, y_pred_lgbm)
report = classification_report(y_test_lgbm, y_pred_lgbm, target_names=species_unique)
logging_with_display(f"LGBM Accuracy: {accuracy:.4f}")
logging_with_display(f"LGBM Classification Report:\n{report}")
cm = confusion_matrix(y_test_lgbm, y_pred_lgbm)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=species_unique)
disp.plot()
plt.title("LGBM Confusion Matrix")
plt.show()